# Notebook 08 — Gaussian Processes: Bayesian Priors Over Functions

## Background

All the models so far have priors over *parameters*: scalars or vectors of finite dimension. A Gaussian Process (GP) is something more powerful — a prior over *functions*. Instead of saying "I believe the slope is probably near 2", a GP says "I believe the underlying function is probably smooth" — without specifying a parametric form.

Formally: a GP is a collection of random variables such that every finite subset is jointly Gaussian. A GP is completely specified by:
- A **mean function** `m(x) = E[f(x)]` — prior belief about the function value
- A **kernel (covariance) function** `k(x, x') = Cov[f(x), f(x')]` — prior belief about how much nearby function values co-vary

The kernel is the key design choice. It encodes beliefs about:
- **Smoothness**: how rapidly can the function change?
- **Lengthscale**: how far apart do points need to be before they're effectively independent?
- **Periodicity**: does the function repeat?
- **Trend**: is there a systematic drift?

## What You'll Learn

- GP intuition: samples from a GP prior, and how the kernel shapes them
- Common kernels: RBF (squared exponential), Matern, periodic, linear
- GP regression: the posterior over functions given noisy observations
- GP in PyMC: `pm.gp.Latent`, `pm.gp.Marginal`, kernel composition
- Practical GP regression and binary classification
- Scalability: why GPs are `O(n^3)` and what to do about it

## Why This Matters

GPs are the Bayesian non-parametric regression workhorse. They're used for:
- Time series modeling when you don't know the functional form
- Bayesian optimization (modeling the objective function before querying it)
- Surrogate modeling for expensive simulations
- Any regression task where you want uncertainty quantification without assuming a parametric model

Scikit-learn includes GPR; PyMC's GP module gives you full Bayesian inference over kernel hyperparameters — not just maximum likelihood estimates.

In [ ]:
import pymc as pm
import arviz as az
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

print(f'PyMC {pm.__version__}  |  ArviZ {az.__version__}')

## Part 1 — GP Priors: Sampling from a Distribution Over Functions

Before seeing any data, the GP prior is a prior over functions. Sampling from a GP prior means: pick a set of x-locations, compute their covariance matrix using the kernel, and draw a multivariate Gaussian sample. Each sample is a plausible function.

In [ ]:
np.random.seed(42)

def rbf_kernel(X1, X2, ell=1.0, sigma_f=1.0):
    """Squared exponential (RBF) kernel.
    k(x, x') = sigma_f^2 * exp(-||x-x'||^2 / (2*ell^2))
    ell: lengthscale -- how far apart before correlation drops
    sigma_f: output scale -- amplitude of function variations
    """
    sq_dist = np.sum((X1[:, None] - X2[None, :])**2, axis=-1)
    return sigma_f**2 * np.exp(-sq_dist / (2 * ell**2))

def matern_52_kernel(X1, X2, ell=1.0, sigma_f=1.0):
    """Matern 5/2 kernel: rougher than RBF but still smooth.
    Allows functions that are twice-differentiable but not infinitely smooth.
    """
    d = np.sqrt(np.sum((X1[:, None] - X2[None, :])**2, axis=-1) + 1e-12)
    z = np.sqrt(5) * d / ell
    return sigma_f**2 * (1 + z + z**2 / 3) * np.exp(-z)

def periodic_kernel(X1, X2, ell=1.0, period=1.0, sigma_f=1.0):
    """Periodic kernel: k(x,x') = sigma_f^2 * exp(-2*sin^2(pi*d/period)/ell^2)"""
    d = np.abs(X1[:, None] - X2[None, :])
    return sigma_f**2 * np.exp(-2 * np.sin(np.pi * d / period)**2 / ell**2)

# Sample from GP priors with different kernels
x_prior = np.linspace(-5, 5, 200).reshape(-1, 1)
n_samples = 5

kernel_configs = [
    ('RBF (ell=0.5, smooth)',  rbf_kernel(x_prior, x_prior, ell=0.5),    'steelblue'),
    ('RBF (ell=2.0, smooth)',  rbf_kernel(x_prior, x_prior, ell=2.0),    'darkorange'),
    ('Matern 5/2 (ell=1.0)',   matern_52_kernel(x_prior, x_prior, ell=1.0), 'seagreen'),
    ('Periodic (p=2)',         periodic_kernel(x_prior, x_prior, period=2.0), 'red'),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for ax, (label, K, color) in zip(axes.flatten(), kernel_configs):
    # Add small jitter for numerical stability
    K_jitter = K + 1e-6 * np.eye(len(x_prior))
    L = np.linalg.cholesky(K_jitter)  # Cholesky for efficient sampling
    
    for i in range(n_samples):
        f_sample = L @ np.random.randn(len(x_prior))
        ax.plot(x_prior.flatten(), f_sample, alpha=0.6, lw=1.2, color=color)
    
    # Mean +/- 2 std from the prior
    prior_std = np.sqrt(np.diag(K))
    ax.fill_between(x_prior.flatten(), -2*prior_std, 2*prior_std,
                    alpha=0.15, color=color, label='95% prior band')
    ax.axhline(0, color='black', linestyle='--', lw=0.8, alpha=0.5)
    ax.set_title(label, fontsize=10)
    ax.set_xlabel('x'); ax.set_ylabel('f(x)')
    ax.legend(fontsize=8)

plt.suptitle('GP Prior Samples: The Kernel Encodes Beliefs About Function Shape', fontsize=12)
plt.tight_layout()
plt.show()

print('Kernel choice encodes your prior beliefs:')
print('  RBF (short ell): rapidly varying, high frequency functions')
print('  RBF (long ell):  slowly varying, smooth functions')
print('  Matern 5/2:      smooth but allows sharper features than RBF')
print('  Periodic:        functions that repeat with a known period')

## Part 2 — GP Posterior: Updating with Observations

Given observations `y = f(X) + noise`, the GP posterior is also a GP — the prior updates via the multivariate Gaussian conditioning formula:

```
mu_post(x*) = K(x*, X) [K(X, X) + sigma_n^2 I]^{-1} y
K_post(x*, x*) = K(x*, x*) - K(x*, X) [K(X, X) + sigma_n^2 I]^{-1} K(X, x*)
```

The posterior mean is a weighted sum of observations (with the kernel providing the weights), and the posterior variance shrinks to zero at observed points.

In [ ]:
def gp_posterior(x_train, y_train, x_test, kernel_fn, noise_std=0.2, **kernel_kwargs):
    """Exact GP posterior under Gaussian noise."""
    n = len(x_train)
    
    # Covariance matrices
    K_tt = kernel_fn(x_train, x_train, **kernel_kwargs) + noise_std**2 * np.eye(n)
    K_ss = kernel_fn(x_test,  x_test,  **kernel_kwargs)
    K_st = kernel_fn(x_test,  x_train, **kernel_kwargs)
    
    # Solve K_tt @ alpha = y via Cholesky (numerically stable)
    L = np.linalg.cholesky(K_tt + 1e-8 * np.eye(n))
    alpha = np.linalg.solve(L.T, np.linalg.solve(L, y_train))
    
    mu_post = K_st @ alpha
    v = np.linalg.solve(L, K_st.T)
    K_post = K_ss - v.T @ v
    std_post = np.sqrt(np.clip(np.diag(K_post), 0, None))
    
    return mu_post, std_post, K_post

# True function and noisy observations
np.random.seed(42)
x_true = np.linspace(-5, 5, 300)
f_true = np.sin(x_true) + 0.3 * np.cos(3 * x_true)

# Sparse observations
x_obs_flat = np.array([-4, -2.5, -1, 0, 1.5, 3, 4.5])
y_obs = np.sin(x_obs_flat) + 0.3 * np.cos(3 * x_obs_flat) + np.random.normal(0, 0.2, len(x_obs_flat))

x_obs = x_obs_flat.reshape(-1, 1)
x_test = x_true.reshape(-1, 1)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

configs = [
    ('RBF ell=0.5',  rbf_kernel,       {'ell': 0.5,  'sigma_f': 1.0}),
    ('RBF ell=1.5',  rbf_kernel,       {'ell': 1.5,  'sigma_f': 1.0}),
    ('Matern 5/2',   matern_52_kernel, {'ell': 1.0,  'sigma_f': 1.0}),
]

for ax, (label, kfn, kwargs) in zip(axes, configs):
    mu, std, _ = gp_posterior(x_obs, y_obs, x_test, kfn, noise_std=0.2, **kwargs)
    
    ax.fill_between(x_true, mu - 2*std, mu + 2*std, alpha=0.2, color='steelblue', label='95% CI')
    ax.fill_between(x_true, mu - std, mu + std,     alpha=0.35, color='steelblue', label='68% CI')
    ax.plot(x_true, mu, 'steelblue', lw=2, label='Posterior mean')
    ax.plot(x_true, f_true, 'r--', lw=1.5, alpha=0.8, label='True function')
    ax.scatter(x_obs_flat, y_obs, color='black', s=60, zorder=5, label='Observations')
    ax.set_title(label, fontsize=11)
    ax.set_xlabel('x'); ax.set_ylabel('f(x)')
    ax.legend(fontsize=7)

plt.suptitle('GP Posterior: Uncertainty Collapses at Observations\n'
             'Between observations: uncertainty determined by kernel lengthscale', fontsize=12)
plt.tight_layout()
plt.show()

## Part 3 — GP Regression in PyMC

In practice we don't fix the kernel hyperparameters — we infer them from data. PyMC's `pm.gp.Marginal` module handles this: you specify the kernel, put priors on its hyperparameters (lengthscale, amplitude, noise), and NUTS samples the posterior over hyperparameters.

In [ ]:
# GP regression in PyMC with Bayesian kernel hyperparameter inference
np.random.seed(42)

# Time series: monthly website traffic with seasonal pattern
n_obs = 36   # 3 years of monthly data
t_obs = np.arange(n_obs, dtype=float)
true_traffic = 500 + 50 * t_obs + 200 * np.sin(2 * np.pi * t_obs / 12) + \
               np.random.normal(0, 30, n_obs)

# Normalize for GP
t_mean, t_std = t_obs.mean(), t_obs.std()
y_mean, y_std = true_traffic.mean(), true_traffic.std()

t_z = (t_obs - t_mean) / t_std
y_z = (true_traffic - y_mean) / y_std

print(f'Website traffic dataset: {n_obs} monthly observations')
print(f'Traffic range: {true_traffic.min():.0f} -- {true_traffic.max():.0f}')

with pm.Model() as gp_model:
    # Kernel hyperparameter priors
    ell     = pm.InverseGamma('ell',     alpha=5, beta=5)   # lengthscale (normalized)
    sigma_f = pm.HalfNormal('sigma_f',   sigma=2)           # output amplitude
    sigma_n = pm.HalfNormal('sigma_n',   sigma=0.5)         # noise level
    
    # RBF kernel (smooth trend) + small noise
    cov = sigma_f**2 * pm.gp.cov.ExpQuad(1, ls=ell)
    gp  = pm.gp.Marginal(cov_func=cov)
    
    # Condition on observations (integrates out the latent function analytically)
    y_obs_gp = gp.marginal_likelihood(
        'y_obs',
        X=t_z[:, None],
        y=y_z,
        noise=sigma_n
    )
    
    trace_gp = pm.sample(
        draws=1000, tune=1000, chains=4,
        target_accept=0.9,
        random_seed=42,
        progressbar=True
    )

print(az.summary(trace_gp, var_names=['ell', 'sigma_f', 'sigma_n']).to_string())

In [ ]:
# Predict future traffic (months 36-47)
t_future = np.arange(n_obs, n_obs + 12, dtype=float)
t_future_z = (t_future - t_mean) / t_std

with gp_model:
    # Conditional distribution: GP posterior at new x values
    gp_pred = gp.conditional('gp_pred', Xnew=t_future_z[:, None])
    pred_samples = pm.sample_posterior_predictive(
        trace_gp, var_names=['gp_pred'], random_seed=42
    )

# Back-transform to original scale
pred_z = pred_samples.posterior_predictive['gp_pred'].values.reshape(-1, 12)
pred_orig = pred_z * y_std + y_mean

fig, ax = plt.subplots(figsize=(13, 6))

# Historical data
ax.plot(t_obs, true_traffic, 'k-', lw=1.5, alpha=0.7, label='Observed traffic')
ax.scatter(t_obs, true_traffic, color='black', s=20, zorder=4)

# Future predictions
pred_mean = pred_orig.mean(axis=0)
pred_lo, pred_hi = np.percentile(pred_orig, [3, 97], axis=0)
pred_lo25, pred_hi75 = np.percentile(pred_orig, [25, 75], axis=0)

ax.fill_between(t_future, pred_lo, pred_hi, alpha=0.2, color='steelblue', label='94% PI')
ax.fill_between(t_future, pred_lo25, pred_hi75, alpha=0.35, color='steelblue', label='50% PI')
ax.plot(t_future, pred_mean, 'steelblue', lw=2.5, label='Posterior mean forecast')
ax.axvline(n_obs - 0.5, color='gray', linestyle='--', alpha=0.7)
ax.text(n_obs, true_traffic.mean(), ' Forecast start', color='gray', fontsize=9)

ax.set_xlabel('Month')
ax.set_ylabel('Monthly traffic')
ax.set_title('GP Regression: Website Traffic Forecasting\n'
             'Kernel hyperparameters inferred via MCMC')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## Part 4 — Kernel Composition

Kernels can be composed by addition and multiplication. This is powerful: you can encode specific beliefs about different aspects of the function.

- **Sum** `k1 + k2`: function is a sum of two independent GPs (e.g., trend + periodic)
- **Product** `k1 * k2`: one component modulates the other (e.g., slowly varying amplitude on a periodic signal)

This is the basis of the 'automatic statistician' idea: systematically search over kernel compositions to find the best structural description of time series.

In [ ]:
# Kernel composition: trend (RBF) + seasonal (periodic * RBF for decay)
# This is a standard time series decomposition via kernel design

np.random.seed(42)
n_ts = 60  # 5 years monthly
t_ts = np.arange(n_ts, dtype=float)

# True components: linear trend + annual seasonality + noise
trend = 0.8 * t_ts
seasonal = 30 * np.sin(2 * np.pi * t_ts / 12)
sales = trend + seasonal + np.random.normal(0, 5, n_ts)

t_ts_z = (t_ts - t_ts.mean()) / t_ts.std()
y_ts_z = (sales - sales.mean()) / sales.std()

with pm.Model() as gp_composed:
    # Component 1: smooth long-term trend
    ell_trend  = pm.InverseGamma('ell_trend', alpha=5, beta=5)
    amp_trend  = pm.HalfNormal('amp_trend', sigma=2)
    cov_trend  = amp_trend**2 * pm.gp.cov.ExpQuad(1, ls=ell_trend)
    
    # Component 2: periodic seasonal pattern
    period     = pm.Normal('period', mu=1.0, sigma=0.2)  # normalized period ~1 year
    ell_per    = pm.InverseGamma('ell_per', alpha=3, beta=3)
    amp_per    = pm.HalfNormal('amp_per', sigma=2)
    cov_per    = amp_per**2 * pm.gp.cov.Periodic(1, period=period, ls=ell_per)
    
    # Composed kernel: trend + seasonal
    cov_total = cov_trend + cov_per
    
    # Noise
    sigma_n = pm.HalfNormal('sigma_n', sigma=0.5)
    
    gp_comp = pm.gp.Marginal(cov_func=cov_total)
    y_obs   = gp_comp.marginal_likelihood(
        'y_obs', X=t_ts_z[:, None], y=y_ts_z, noise=sigma_n
    )
    
    trace_comp = pm.sample(
        draws=1000, tune=1000, chains=2,
        target_accept=0.9, random_seed=42, progressbar=True
    )

print(az.summary(trace_comp, var_names=['ell_trend', 'amp_trend', 'ell_per', 'amp_per', 'period']).to_string())

In [ ]:
# Forecast next 24 months
t_future_ts = np.arange(n_ts, n_ts + 24, dtype=float)
t_future_ts_z = (t_future_ts - t_ts.mean()) / t_ts.std()

with gp_composed:
    gp_future = gp_comp.conditional('gp_future', Xnew=t_future_ts_z[:, None])
    pred_comp = pm.sample_posterior_predictive(
        trace_comp, var_names=['gp_future'], random_seed=42
    )

pred_comp_z = pred_comp.posterior_predictive['gp_future'].values.reshape(-1, 24)
pred_comp_orig = pred_comp_z * sales.std() + sales.mean()

fig, ax = plt.subplots(figsize=(13, 6))
ax.plot(t_ts, sales, 'k-', lw=1.5, alpha=0.8, label='Historical sales')
ax.scatter(t_ts, sales, color='black', s=15, zorder=4)

pred_mean_c  = pred_comp_orig.mean(axis=0)
pred_lo_c, pred_hi_c = np.percentile(pred_comp_orig, [3, 97], axis=0)
ax.fill_between(t_future_ts, pred_lo_c, pred_hi_c, alpha=0.2, color='seagreen', label='94% PI')
ax.plot(t_future_ts, pred_mean_c, 'seagreen', lw=2.5, label='Composed GP forecast')
ax.axvline(n_ts - 0.5, color='gray', linestyle='--', alpha=0.7)
ax.set_xlabel('Month'); ax.set_ylabel('Sales')
ax.set_title('Composed GP Kernel: Trend + Seasonal\n'
             'cov = RBF (trend) + Periodic (seasonality)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## Part 5 — Scalability and Approximate GPs

Exact GP regression requires solving an `n x n` linear system — `O(n^3)` time and `O(n^2)` memory. For `n = 10,000` this is already slow; for `n = 100,000` it's infeasible.

Modern approaches for large-scale GPs:
- **Sparse GPs** (FITC, VFE): introduce `m << n` inducing points; `O(m^2 n)` complexity
- **State-space GPs**: exploit Markov structure for time series; `O(n)` for Matern kernels
- **Random Fourier features**: approximate the kernel with finite basis; `O(nd)` with `d` features
- **Deep kernel learning**: neural network + GP; learns feature representation automatically

PyMC supports sparse GPs via `pm.gp.MarginalSparse` with inducing points. For most practical datasets (n < 5000), exact GPs are fine.

In [ ]:
# Illustrate: how GP complexity scales with n
import time

print('GP Scaling Benchmark (exact posterior computation):')
print(f'{"n":>8} {"Time (ms)":>12} {"Memory (MB)":>14}')
print('-' * 38)

for n in [100, 500, 1000, 2000, 5000]:
    x_bench = np.random.rand(n, 1)
    K = rbf_kernel(x_bench, x_bench, ell=0.3) + 1e-4 * np.eye(n)
    
    start = time.perf_counter()
    L = np.linalg.cholesky(K)
    _ = np.linalg.solve(L.T, np.linalg.solve(L, np.random.randn(n)))
    elapsed_ms = (time.perf_counter() - start) * 1000
    
    mem_mb = (n * n * 8) / 1e6  # float64 covariance matrix
    print(f'{n:>8} {elapsed_ms:>12.1f} {mem_mb:>14.1f}')

print()
print('O(n^3) compute, O(n^2) memory.')
print('For n > 5000: use pm.gp.MarginalSparse with inducing points.')

## Summary

| Component | Description | Design choice |
|-----------|-------------|---------------|
| Mean function | Prior belief about f(x) | Usually zero; can encode trend |
| Kernel | Encodes smoothness, periodicity | RBF=smooth, Matern=rougher, Periodic=cyclic |
| Noise model | Observation model | Gaussian (regression), Bernoulli (classification) |
| Hyperparameters | Lengthscale, amplitude | Put priors on these; NUTS infers them |

**PyMC GP API**:
```python
with pm.Model():
    ell   = pm.InverseGamma('ell', alpha=5, beta=5)
    sigma = pm.HalfNormal('sigma', sigma=2)
    cov   = sigma**2 * pm.gp.cov.ExpQuad(1, ls=ell)  # RBF kernel
    gp    = pm.gp.Marginal(cov_func=cov)
    y_obs = gp.marginal_likelihood('y', X=X_train, y=y_train, noise=noise)
    trace = pm.sample(draws=1000, tune=1000)

    # Posterior predictions
    f_star = gp.conditional('f_star', Xnew=X_test)
    ppc = pm.sample_posterior_predictive(trace, var_names=['f_star'])
```

**Kernel composition**: `cov = cov1 + cov2` (additive) or `cov1 * cov2` (multiplicative)

**Scalability**: exact GPs are O(n^3); for n > 5000 use `pm.gp.MarginalSparse`

**Next**: Notebook 09 — Variational Inference. ELBO maximization, mean-field VI, and PyMC's ADVI — a faster alternative to NUTS for large models.